``learning_curve`` shows whether a prediction task is sampling-limited. It repeats the stratified cross-validation of ``run`` on stratified subsets of increasing size of every training fold and scores each model on the full, unchanged test fold. First, the ``DOM_GSEC`` dataset and its feature matrix:

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False  # Disable verbosity

# DOM_GSEC example dataset + a small feature set (see [Breimann25]_)
df_seq = aa.load_dataset(name="DOM_GSEC")
labels = df_seq["label"].to_list()
df_feat = aa.load_features(name="DOM_GSEC").head(20)

# Build the CPP feature matrix X
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
X = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts)

Pass ``X`` and ``labels`` with the training-subset sizes (``train_sizes``, fractions of the smallest training fold or absolute sample counts), the number of folds (``n_cv``), the number of cross-validation repeats (``n_rounds``), the ``metrics``, the bootstrap confidence level ``ci``, and a per-call ``random_state``. The result has one row per (model, training size, metric):

In [2]:
me = aa.ModelEvaluator(models=["rf", "log_reg"], random_state=42, verbose=False)
df_curve = me.learning_curve(X=X, labels=labels, train_sizes=[0.1, 0.25, 0.5, 0.75, 1.0],
                             n_cv=5, n_rounds=3, metrics=["balanced_accuracy", "mcc"],
                             ci=0.95, random_state=42)
aa.display_df(df_curve, n_rows=10, show_shape=True)

DataFrame shape: (20, 8)


,model,train_size,metric,score,score_std,ci_low,ci_high,n_scores
1,rf,10,balanced_accuracy,0.790812,0.092653,0.742089,0.833974,15
2,rf,10,mcc,0.597695,0.181774,0.503134,0.682840,15
3,rf,25,balanced_accuracy,0.789316,0.097634,0.740807,0.837399,15
4,rf,25,mcc,0.590024,0.192078,0.492990,0.685081,15
5,rf,50,balanced_accuracy,0.804701,0.092453,0.758114,0.845315,15
6,rf,50,mcc,0.617305,0.186296,0.523176,0.699471,15
7,rf,75,balanced_accuracy,0.801282,0.100295,0.751923,0.847222,15
8,rf,75,mcc,0.607160,0.199625,0.509310,0.699319,15
9,rf,100,balanced_accuracy,0.803632,0.076990,0.763862,0.838034,15
10,rf,100,mcc,0.612956,0.151909,0.534722,0.680772,15


A score that still rises at the largest ``train_size`` suggests that more data will help, while a flat curve suggests changing the representation or model. Absolute sample counts can be given instead of fractions, and ``ci=None`` skips the bootstrap:

In [3]:
df_curve = me.learning_curve(X, labels, train_sizes=[10, 20, 40, 80], metrics=["mcc"], ci=None)
aa.display_df(df_curve, n_rows=10, show_shape=True)

DataFrame shape: (8, 8)


,model,train_size,metric,score,score_std,ci_low,ci_high,n_scores
1,rf,10,mcc,0.580965,0.246694,nan,nan,5
2,rf,20,mcc,0.590384,0.185111,nan,nan,5
3,rf,40,mcc,0.587075,0.119189,nan,nan,5
4,rf,80,mcc,0.575011,0.122296,nan,nan,5
5,log_reg,10,mcc,0.608698,0.188068,nan,nan,5
6,log_reg,20,mcc,0.669767,0.167276,nan,nan,5
7,log_reg,40,mcc,0.668534,0.164187,nan,nan,5
8,log_reg,80,mcc,0.621766,0.139291,nan,nan,5
